# Модель C: скрининг-классификатор include/exclude (обучение в YC, plan §7)

bi-encoder эмбеддинг (e5-small) пары `PICO [SEP] title. abstract` + LR-голова. Цель — **recall >= 0.95** (не терять РКИ): в пайплайне работает как recall-first второе мнение поверх LLM-скрининга (`src/adapters/screen/biencoder_clf.py`, `ScreeningPort`).

**Данные — реальные, из CLEF eHealth TAR 2019** (открытый бенчмарк скрининга систематических обзоров, ручная разметка include/exclude авторами Cochrane-обзоров). Сборка: `training/build_screening_dataset.py` → `screening_labels.csv`. Метки: `include` / `exclude`. Адаптер использует `predict_proba` + низкий порог (`threshold`) → высокий recall. **Чекпойнт:** `screening_clf.joblib` → Object Storage → `data/models/screening_clf.joblib`.

In [ ]:
# %pip install -q sentence-transformers scikit-learn joblib

In [ ]:
# РЕАЛЬНЫЕ данные из CLEF eHealth TAR 2019 (ручная разметка include/exclude авторами
# Cochrane-обзоров). Сборка: `training/build_screening_dataset.py` -> screening_labels.csv
# (query,doc,label). Если CSV нет — крошечный демо-сэмпл.
import csv
from pathlib import Path

SEP = " [SEP] "
csv_path = Path("screening_labels.csv")
if csv_path.exists():
    with csv_path.open(encoding="utf-8") as f:
        rows = [(r["query"], r["doc"], r["label"]) for r in csv.DictReader(f)]
    print(f"реальные данные: {len(rows)} строк из {csv_path}")
else:
    rows = [
        (
            "melanoma BRAF+MEK vs BRAF overall survival",
            "Dabrafenib plus trametinib vs dabrafenib in melanoma. OS hazard ratio 0.71.",
            "include",
        ),
        (
            "melanoma BRAF+MEK vs BRAF overall survival",
            "Cost-effectiveness analysis of targeted therapy in melanoma.",
            "exclude",
        ),
        (
            "HER2+ breast cancer trastuzumab overall survival",
            "Adjuvant trastuzumab vs chemotherapy alone in HER2+ EBC. Survival improved.",
            "include",
        ),
        (
            "HER2+ breast cancer trastuzumab overall survival",
            "Review of cardiotoxicity monitoring guidelines.",
            "exclude",
        ),
    ]
    print(f"демо-сэмпл: {len(rows)} строк (положи screening_labels.csv для реальных метрик)")
texts = [q + SEP + doc for q, doc, _ in rows]
labels = [y for _, _, y in rows]
len(texts)

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("intfloat/multilingual-e5-small")
X = encoder.encode(texts, normalize_embeddings=True)
X.shape

In [ ]:
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, recall_score
from sklearn.model_selection import train_test_split

# Честная recall-first оценка на hold-out (крит. 9), не на train=test. class_weight='balanced'
# — скрининг сильно несбалансирован (~1 include на 3-70 exclude). Порог низкий -> recall>=0.95.
threshold = 0.3
min_class = min(Counter(labels).values())
if min_class >= 8:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, labels, test_size=0.25, random_state=0, stratify=labels
    )
    clf_ho = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
    col = list(clf_ho.classes_).index("include")
    proba = clf_ho.predict_proba(X_te)[:, col]
    pred = ["include" if p >= threshold else "exclude" for p in proba]
    print("hold-out recall(include):", recall_score(y_te, pred, pos_label="include"))
    print(classification_report(y_te, pred))
else:
    print("демо: слишком мало данных для hold-out — нужен screening_labels.csv")

In [ ]:
import joblib

# Финальный чекпойнт — на ВСЕХ данных, class_weight='balanced' (как в hold-out-оценке).
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X, labels)
joblib.dump(clf, "screening_clf.joblib")
print("сохранён screening_clf.joblib")
# classes_ включает 'include' -> адаптер берёт P(include) и сравнивает с threshold.
# -> Object Storage -> data/models/screening_clf.joblib (путь адаптера).